# What Drives the Price of a Used Car?

## Executive summary

This CRISP-DM analysis identifies vehicle characteristics associated with used-car listing price. The audience is a used-car dealership deciding which vehicles to acquire and how to price them.


## 1. Business understanding

**Question:** Which observable vehicle features are most associated with used-car price?

**Decision:** Use the results to prioritize inventory and price comparable vehicles consistently. The primary metric is mean absolute error (MAE), because it expresses the typical prediction error in dollars and is less affected by unusually expensive listings than squared-error metrics.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='deep')
DATA = Path('../data/vehicles.csv') if Path('../data/vehicles.csv').exists() else Path('data/vehicles.csv')
raw = pd.read_csv(DATA)
print(f'Raw data shape: {raw.shape}')
raw.head()

## 2. Data understanding and preparation

Identifiers (`id` and `VIN`) are excluded. Listings with nonpositive or implausibly high prices, implausible years, or implausible mileage are removed. Missing feature values are imputed inside the pipeline to avoid data leakage.


In [ ]:
raw.info()
display(raw[['price', 'year', 'odometer']].describe().T)
display(raw.isna().mean().sort_values(ascending=False).head(12))

analysis = raw.loc[raw.price.between(500, 100000) & raw.year.between(1990, 2022) & raw.odometer.between(0, 350000)].copy()
analysis['age'] = 2022 - analysis.year
features = ['age', 'odometer', 'manufacturer', 'condition', 'cylinders', 'fuel', 'title_status', 'transmission', 'drive', 'size', 'type', 'paint_color', 'state']
analysis = analysis[features + ['price']].dropna(subset=['price'])
model_data = analysis.sample(n=min(50000, len(analysis)), random_state=42)
print(f'Clean rows: {len(analysis):,}; modeling sample: {len(model_data):,}')

## 3. Exploratory data analysis

The distribution and comparison plots below provide a nontechnical view of the data. The price charts are limited to $60,000 where noted so typical dealership inventory remains legible.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(model_data.price, bins=60, ax=axes[0], color='#2A6F97')
axes[0].set(xlim=(0, 60000), title='Used-car listing price distribution', xlabel='Price ($)')
sns.scatterplot(data=model_data.sample(min(8000, len(model_data)), random_state=42), x='odometer', y='price', alpha=.25, ax=axes[1])
axes[1].set(ylim=(0, 100000), title='Mileage and listing price', xlabel='Odometer (miles)', ylabel='Price ($)')
top_makes = model_data.manufacturer.value_counts().head(10).index
sns.boxplot(data=model_data[model_data.manufacturer.isin(top_makes)], x='manufacturer', y='price', showfliers=False, ax=axes[2])
axes[2].tick_params(axis='x', rotation=45)
axes[2].set(ylim=(0, 60000), title='Price varies by manufacturer', xlabel='Manufacturer', ylabel='Price ($)')
plt.tight_layout()

## 4. Modeling

Ridge regression is a regularized linear baseline. Random forest can capture nonlinear relationships and interactions. Both models use the same train/test split, 5-fold cross-validation, and GridSearchCV. The held-out test set is used only for final evaluation.


In [ ]:
X, y = model_data[features], model_data.price
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, random_state=42)
numeric = ['age', 'odometer']
categorical = [c for c in features if c not in numeric]
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical)])
cv = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_search = GridSearchCV(Pipeline([('prep', preprocessor), ('model', Ridge())]), {'model__alpha': [1, 10, 100]}, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
forest_search = GridSearchCV(Pipeline([('prep', preprocessor), ('model', RandomForestRegressor(random_state=42, n_jobs=-1))]), {'model__n_estimators': [150], 'model__max_depth': [16, None], 'model__min_samples_leaf': [1, 3]}, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
ridge_search.fit(X_train, y_train)
forest_search.fit(X_train, y_train)
print('Best Ridge:', ridge_search.best_params_, 'CV MAE:', round(-ridge_search.best_score_, 2))
print('Best Forest:', forest_search.best_params_, 'CV MAE:', round(-forest_search.best_score_, 2))

In [ ]:
def evaluate(name, estimator):
    prediction = estimator.predict(X_test)
    return {'Model': name, 'Test MAE ($)': mean_absolute_error(y_test, prediction), 'Test RMSE ($)': mean_squared_error(y_test, prediction) ** .5, 'Test R2': r2_score(y_test, prediction)}

results = pd.DataFrame([evaluate('Ridge regression', ridge_search.best_estimator_), evaluate('Random forest', forest_search.best_estimator_)]).sort_values('Test MAE ($)')
display(results.style.format({'Test MAE ($)': '${:,.0f}', 'Test RMSE ($)': '${:,.0f}', 'Test R2': '{:.3f}'}))
best_forest = forest_search.best_estimator_
importance = pd.Series(best_forest.named_steps['model'].feature_importances_, index=best_forest.named_steps['prep'].get_feature_names_out())
grouped = importance.groupby(lambda x: x.split('__')[1].split('_')[0]).sum().sort_values(ascending=False)
plt.figure(figsize=(9, 5)); sns.barplot(x=grouped.values, y=grouped.index, color='#2A6F97')
plt.title('Model feature importance by vehicle attribute'); plt.xlabel('Relative importance'); plt.ylabel('Feature'); plt.tight_layout()
display(grouped)

## 5. Findings and dealer recommendations

1. **Price the fundamentals first.** Age and mileage should be required inputs to every acquisition and pricing decision.
2. **Buy to segment demand.** Manufacturer, body type, drive, fuel, and transmission differentiate comparable vehicles. Compare potential purchases against their segment, not the overall market average.
3. **Treat title status and condition as risk controls.** Verify clean title and condition before buying because adverse or missing information can reduce consumer willingness to pay.
4. **Use the winning model as a pricing guardrail.** Compare a listing with the prediction, then adjust for inspection findings, reconditioning cost, local competition, and inventory velocity.

### Limitations and next steps

Listing price is not final transaction price. Add days-on-lot, accident history, trim/options, local competition, transaction price, and reconditioning cost. Retrain regularly and monitor MAE by vehicle segment. Feature importance identifies predictive association, not causation.
